In [ ]:
import requests
from pathlib import Path

class APIClient:
    def __init__(self, base_url, username, password):
        self.base_url = base_url
        self.username = username
        self.password = password
        self.access_token = None
        self.refresh_token = None

    # ---------- AUTH ----------
    def authenticate(self):
        url = f"{self.base_url}/auth"
        payload = {
            "username": self.username,
            "password": self.password
        }

        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()

        data = r.json()
        self.access_token = data["access_token"]
        self.refresh_token = data["refresh_token"]

    def refresh(self):
        url = f"{self.base_url}/auth/refresh"
        payload = {
            "refresh_token": self.refresh_token
        }

        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()

        data = r.json()
        self.access_token = data["access_token"]

    # ---------- UTILS ----------
    def headers(self):
        return {
            "Authorization": f"Bearer {self.access_token}"
        }

    def request_with_refresh(self, method, url, **kwargs):
        r = requests.request(method, url, headers=self.headers(), **kwargs)

        if r.status_code == 401:
            print("🔁 Token expiré → refresh")
            self.refresh()
            r = requests.request(method, url, headers=self.headers(), **kwargs)

        return r

    # ---------- UPLOAD ----------
    def upload_image(self, job_id, image_path):
        url = f"{self.base_url}/upload/{job_id}"

        with open(image_path, "rb") as img:
            files = {
                "image": (image_path.name, img, "image/jpeg")
            }
            data = {
                "job_id": job_id   # ⚠️ même valeur que dans l'URL
            }

            r = self.request_with_refresh(
                "POST",
                url,
                files=files,
                data=data,
                timeout=30
            )

        r.raise_for_status()
        return r.json()


In [ ]:
client = APIClient(
    base_url="https://api.exemple.com",
    username="mon_user",
    password="mon_mot_de_passe"
)

client.authenticate()


In [ ]:
IMAGE_DIR = Path("images")

images = sorted(
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.png"))
)

In [ ]:
job_id = "abc123"

results = []

for img_path in images:
    try:
        result = client.upload_image(job_id, img_path)
        results.append(result)
        print(f"✅ {img_path.name} envoyé")
    except Exception as e:
        print(f"❌ {img_path.name} → {e}")
